<a href="https://colab.research.google.com/github/davidrpugh/data-analytics-with-polars/blob/main/ch06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 6: Reading and Writing Data

In [ ]:
import polars as pl
pl.__version__

## Format Overview

## Reading CSV Files

In [ ]:
%%bash

curl -L -o "sample_data/penguins.csv" \
  "https://github.com/davidrpugh/data-analytics-with-polars/raw/refs/heads/main/data/penguins.csv"

In [ ]:
!ls -lh sample_data/penguins.csv

In [ ]:
!head -n 5 sample_data/penguins.csv

In [ ]:
penguins = pl.read_csv("sample_data/penguins.csv")
penguins

## Parsing Missing Values Correctly

In [ ]:
penguins = pl.read_csv("sample_data/penguins.csv", null_values="NA")
penguins

In [ ]:
(
    penguins.null_count()
            .transpose(
                include_header=True,
                column_names=["null_count"]
            )
)

## Reading Files with Encodings Other Than UTF-8

In [ ]:
%%bash

curl -L -o "sample_data/directors.csv" \
  "https://github.com/davidrpugh/data-analytics-with-polars/raw/refs/heads/main/data/directors.csv"

In [ ]:
# This raises a ComputeError:
pl.read_csv("sample_data/directors.csv")

In [ ]:
pl.read_csv("sample_data/directors.csv", encoding="EUC-CN")

In [ ]:
import chardet


def detect_encoding(filename: str) -> str:
    """Return the most probable character encoding for a file."""

    with open(filename, "rb") as f:
        raw_data = f.read()
        result = chardet.detect(raw_data)
        return result["encoding"]


detect_encoding("sample_data/directors.csv")

In [ ]:
pl.read_csv("sample_data/directors.csv", encoding="EUC-JP")

## Reading Excel Spreadsheets

In [ ]:
%%bash

pip install fastexcel

In [ ]:
%%bash

curl -L -o "sample_data/top2000-2023.xlsx" \
  "https://github.com/davidrpugh/data-analytics-with-polars/raw/refs/heads/main/data/top2000-2023.xlsx"

In [ ]:
songs = pl.read_excel("sample_data/top2000-2023.xlsx")
songs

## Working with Multiple Files

### Downloading Multiple CSV Files with Nested Bash Loops

To download multiple files structured under different directories (such as different stock symbols and years), you can use nested `for` loops in a bash cell.

In [ ]:
%%bash

# Define the stocks
STOCKS="asml nvda tsm"

# Nested loops using range expansion for years 1999 to 2023
for stock in $STOCKS; do
  for year in {1999..2023}; do
    # Create target directory if it doesn't exist
    mkdir -p "sample_data/stock/${stock}"

    echo "Downloading ${stock} data for year ${year}..."

    # Download the CSV using curl
    curl -L -o "sample_data/stock/${stock}/${year}.csv" \
      "https://raw.githubusercontent.com/davidrpugh/data-analytics-with-polars/main/data/stock/${stock}/${year}.csv"
  done
done

In [ ]:
pl.read_csv("sample_data/stock/nvda/201?.csv")

In [ ]:
all_stocks = pl.read_csv("sample_data/stock/**/*.csv")
all_stocks

In [ ]:
import calendar


filenames = [
    f"sample_data/stock/asml/{year}.csv"
    for year in range(1999, 2024)
    if calendar.isleap(year)
]

filenames

In [ ]:
pl.concat(pl.read_csv(f) for f in filenames)

## Reading Parquet

In [ ]:
%%bash

curl -L --create-dirs \
  -o "sample_data/taxi/yellow_tripdata_2022-#1.parquet" \
  "https://github.com/davidrpugh/data-analytics-with-polars/raw/refs/heads/main/data/taxi/yellow_tripdata_2022-{01,02,03,04,05,06,07,08,09,10,11,12}.parquet"

In [ ]:
%%time

trips = pl.read_parquet("sample_data/taxi/yellow_tripdata_*.parquet")
trips

## Reading JSON and NDJSON

### JSON

In [ ]:
%%bash

curl -L -o "sample_data/pokedex.json" \
  "https://github.com/davidrpugh/data-analytics-with-polars/raw/refs/heads/main/data/pokedex.json"

In [ ]:
!cat sample_data/pokedex.json

In [ ]:
pokedex = pl.read_json("sample_data/pokedex.json")
pokedex

In [ ]:
(
    pokedex.explode("pokemon")
    .unnest("pokemon")
    .select("id", "name", "type", "height", "weight")
)

### NDJSON

In [ ]:
%%bash

curl -L -o "sample_data/wikimedia.ndjson" \
  "https://github.com/davidrpugh/data-analytics-with-polars/raw/refs/heads/main/data/wikimedia.ndjson"

In [ ]:
!cat sample_data/wikimedia.ndjson

In [ ]:
from json import loads
from pprint import pprint


with open("sample_data/wikimedia.ndjson") as f:
    pprint(loads(f.readline()))

In [ ]:
wikimedia = pl.read_ndjson("sample_data/wikimedia.ndjson")
wikimedia

In [ ]:
(
    wikimedia.rename({"id": "edit_id"})
    .unnest("meta")
    .select("timestamp", "title", "user", "comment")
)

## Other File Formats

In [ ]:
import pandas as pd


url = "https://en.wikipedia.org/wiki/List_of_Latin_abbreviations"
pl.from_pandas(pd.read_html(url, storage_options={"User-Agent": "Mozilla/5.0"})[0])

## Querying Databases

In [ ]:
%%bash

pip install connectorx

In [ ]:
%%bash

curl -L -o "sample_data/sakila.db" \
  "https://github.com/davidrpugh/data-analytics-with-polars/raw/refs/heads/main/data/sakila.db"

In [ ]:
pl.read_database_uri(
    query="""
    SELECT
        f.film_id,
        f.title,
        c.name AS category,
        f.rating,
        f.length / 60.0 AS length
    FROM
        film AS f,
        film_category AS fc,
        category AS c
    WHERE
        fc.film_id = f.film_id
        AND fc.category_id = c.category_id
    LIMIT 10
    """,
    uri="sqlite:::sample_data/sakila.db",
)

In [ ]:
db = "sqlite:::sample_data/sakila.db"
films = pl.read_database_uri("SELECT * FROM film", db)
film_categories = pl.read_database_uri("SELECT * FROM film_category", db)
categories = pl.read_database_uri("SELECT * FROM category", db)

(
    films.join(film_categories, on="film_id", suffix="_fc")
    .join(categories, on="category_id", suffix="_c")
    .select(
        "film_id",
        "title",
        pl.col("name").alias("category"),
        "rating",
        pl.col("length") / 60,
    )
    .limit(10)
)

## Writing Data

### CSV Format

In [ ]:
all_stocks.write_csv("sample_data/all_stocks.csv")

In [ ]:
!ls -lh sample_data/all_stocks.csv

### Excel Format

In [ ]:
%%bash

pip install xlsxwriter

In [ ]:
all_stocks.write_excel("sample_data/all_stocks.xlsx")

In [ ]:
!ls -lh sample_data/all_stocks.xlsx

### Parquet Format

In [ ]:
all_stocks.write_parquet("sample_data/all_stocks.parquet")

In [ ]:
!ls -lh sample_data/all_stocks.parquet

In [ ]:
!ls -lh sample_data/all_stocks.*